# Phase 3 — Augmentation Pipeline

Two techniques applied to the existing 203-image training set:
1. **Copy-Paste augmentation** — cuts real objects from training images and pastes them onto other images
2. **Strong albumentations pipeline** — geometric + photometric transforms that cover Tier 2 improvements

Run all cells top to bottom. Output goes to `../dataset_augmented/` so the original `../dataset/` is untouched.

In [ ]:
import os, sys, random, shutil, warnings
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from tqdm import tqdm
warnings.filterwarnings('ignore')

# albumentations — install if missing
try:
    import albumentations as A
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations'])
    import albumentations as A

print(f'albumentations {A.__version__}')
print('Imports OK')

## Configuration

In [ ]:
DATASET_DIR  = '../dataset'           # source dataset (never modified)
OUTPUT_DIR   = '../dataset_augmented' # augmented output
IMG_SIZE     = 640
RANDOM_SEED  = 42
AUGMENT_MULT = 3    # generate 3x the original training images via albumentations
CP_PER_IMAGE = 2    # objects to copy-paste into each image
MAX_OVERLAP  = 0.25 # max IoU allowed between pasted object and existing boxes
MIN_OBJ_PX   = 32  # minimum crop dimension in pixels to be worth pasting

CLASS_NAMES = ['dumbbell', 'barbell', 'kettlebell', 'resistance_band', 'pull_up_bar']
CLASS_COLORS = [(220,80,80),(80,200,80),(80,120,220),(230,175,30),(160,80,210)]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f'Source : {DATASET_DIR}')
print(f'Output : {OUTPUT_DIR}')
print(f'Aug multiplier : {AUGMENT_MULT}x  |  Copy-paste per image : {CP_PER_IMAGE}')

## Helper Functions

In [ ]:
def load_labels(label_path):
    """Read YOLO labels → list of [cls, cx, cy, w, h] (normalised)."""
    labels = []
    path = Path(label_path)
    if path.exists() and path.stat().st_size > 0:
        for line in path.read_text().strip().splitlines():
            parts = list(map(float, line.split()))
            if len(parts) == 5:
                labels.append(parts)
    return labels

def save_labels(labels, label_path):
    """Write YOLO labels to file."""
    with open(label_path, 'w') as f:
        for lb in labels:
            cls = int(lb[0])
            f.write(f'{cls} {lb[1]:.6f} {lb[2]:.6f} {lb[3]:.6f} {lb[4]:.6f}\n')

def yolo_to_pixel(cx, cy, bw, bh, W, H):
    """Normalised YOLO → pixel [x1,y1,x2,y2]."""
    x1 = int((cx - bw/2) * W); y1 = int((cy - bh/2) * H)
    x2 = int((cx + bw/2) * W); y2 = int((cy + bh/2) * H)
    return max(0,x1), max(0,y1), min(W,x2), min(H,y2)

def pixel_to_yolo(x1, y1, x2, y2, W, H):
    """Pixel [x1,y1,x2,y2] → normalised YOLO cx,cy,bw,bh."""
    cx = (x1+x2) / 2 / W; cy = (y1+y2) / 2 / H
    bw = (x2-x1) / W;      bh = (y2-y1) / H
    return cx, cy, bw, bh

def draw_boxes(img, labels, title=''):
    """Draw YOLO bounding boxes on image for visualisation."""
    H, W = img.shape[:2]
    fig, ax = plt.subplots(1,1,figsize=(6,6))
    ax.imshow(img)
    for lb in labels:
        cls = int(lb[0])
        x1,y1,x2,y2 = yolo_to_pixel(*lb[1:], W, H)
        color = [c/255 for c in CLASS_COLORS[cls]]
        rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, max(0,y1-4), CLASS_NAMES[cls], color=color,
                fontsize=8, fontweight='bold', backgroundcolor='white')
    ax.set_title(title); ax.axis('off')
    plt.tight_layout(); plt.show()

def load_image(path):
    """Load image as RGB numpy array."""
    img = cv2.imread(str(path))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None

print('Helper functions defined.')

## Step 1 — Build Object Pool

Extract every labelled object crop from the training set.
These crops become the "stamps" for copy-paste augmentation.

In [ ]:
def build_object_pool(images_dir, labels_dir):
    """
    Returns a list of dicts: {'class_id': int, 'crop': H×W×3 RGB array}
    One entry per bounding box across all training images.
    """
    pool = []
    img_files = sorted(
        list(Path(images_dir).glob('*.jpg')) +
        list(Path(images_dir).glob('*.jpeg')) +
        list(Path(images_dir).glob('*.png'))
    )
    for img_file in tqdm(img_files, desc='Building pool'):
        img = load_image(img_file)
        if img is None:
            continue
        H, W = img.shape[:2]
        label_file = Path(labels_dir) / (img_file.stem + '.txt')
        for lb in load_labels(label_file):
            cls = int(lb[0])
            x1, y1, x2, y2 = yolo_to_pixel(*lb[1:], W, H)
            if (x2 - x1) >= MIN_OBJ_PX and (y2 - y1) >= MIN_OBJ_PX:
                pool.append({'class_id': cls, 'crop': img[y1:y2, x1:x2].copy()})

    # Print pool composition
    print(f'\nObject pool: {len(pool)} crops total')
    counts = {}
    for obj in pool:
        counts[obj['class_id']] = counts.get(obj['class_id'], 0) + 1
    for cls_id, cnt in sorted(counts.items()):
        print(f'  {CLASS_NAMES[cls_id]:20s}: {cnt} crops')
    return pool

object_pool = build_object_pool(
    f'{DATASET_DIR}/images/train',
    f'{DATASET_DIR}/labels/train'
)

## Step 2 — Copy-Paste Augmentation

Algorithm:
1. For each training image, randomly pick `CP_PER_IMAGE` crops from the pool
2. Randomly scale each crop (50%–120% of original size)
3. Find a random position that does not heavily overlap existing boxes
4. Paste the crop; add the new bounding box to the label list

In [ ]:
def box_iou(a, b):
    """IoU between two [x1,y1,x2,y2] boxes."""
    ix1, iy1 = max(a[0],b[0]), max(a[1],b[1])
    ix2, iy2 = min(a[2],b[2]), min(a[3],b[3])
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter  = (ix2-ix1) * (iy2-iy1)
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-6)

def copy_paste(img, labels, pool, num_paste=2, max_attempts=20):
    """
    Paste `num_paste` random objects from `pool` onto `img`.
    Returns (augmented_img, updated_labels).
    """
    img   = img.copy()
    H, W  = img.shape[:2]
    labs  = [list(lb) for lb in labels]

    # Convert existing labels to pixel boxes for IoU checks
    existing = [list(yolo_to_pixel(*lb[1:], W, H)) for lb in labs]

    pasted = 0
    for _ in range(num_paste * max_attempts):
        if pasted >= num_paste:
            break

        obj  = random.choice(pool)
        crop = obj['crop'].copy()
        oh, ow = crop.shape[:2]
        if oh < MIN_OBJ_PX or ow < MIN_OBJ_PX:
            continue

        # Random scale
        scale  = random.uniform(0.5, 1.2)
        nw, nh = max(MIN_OBJ_PX, int(ow*scale)), max(MIN_OBJ_PX, int(oh*scale))
        if nw >= W or nh >= H:
            continue
        crop = cv2.resize(crop, (nw, nh))

        # Random flip
        if random.random() < 0.5:
            crop = cv2.flip(crop, 1)

        # Random position
        x1 = random.randint(0, W - nw)
        y1 = random.randint(0, H - nh)
        x2, y2 = x1 + nw, y1 + nh
        new_box = [x1, y1, x2, y2]

        # Reject if too much overlap with existing boxes
        if any(box_iou(new_box, eb) > MAX_OVERLAP for eb in existing):
            continue

        # Paste
        img[y1:y2, x1:x2] = crop
        existing.append(new_box)

        cx, cy, bw, bh = pixel_to_yolo(x1, y1, x2, y2, W, H)
        labs.append([float(obj['class_id']), cx, cy, bw, bh])
        pasted += 1

    return img, labs

print('copy_paste() defined.')

## Step 3 — Strong Albumentations Pipeline

Covers all Tier 2 techniques:
- **Geometric**: flip, perspective, affine scale, random crop
- **Photometric**: brightness/contrast, HSV shift, colour jitter, CLAHE
- **Noise/blur**: Gaussian blur, Gaussian noise, motion blur
- **Occlusion**: CoarseDropout (simulates partial occlusion)

All spatial transforms automatically adjust bounding boxes via `BboxParams`.

In [ ]:
def build_aug_pipeline(img_size=IMG_SIZE):
    # Check albumentations version for API compatibility
    ver = tuple(int(x) for x in A.__version__.split('.')[:2])

    coarse = (
        A.CoarseDropout(num_holes_range=(1,4), hole_height_range=(20,60),
                        hole_width_range=(20,60), p=0.30)
        if ver >= (1,4)
        else A.CoarseDropout(max_holes=4, max_height=60, max_width=60,
                             min_holes=1, min_height=20, min_width=20, p=0.30)
    )
    gnoise = (
        A.GaussNoise(std_range=(0.02, 0.08), p=0.30)
        if ver >= (1,4)
        else A.GaussNoise(var_limit=(10.0, 50.0), p=0.30)
    )

    return A.Compose([
        # ── Geometric ──────────────────────────────────────────
        A.HorizontalFlip(p=0.50),
        A.VerticalFlip(p=0.15),
        A.Affine(scale=(0.75, 1.25), shear=(-8, 8),
                 rotate=(-15, 15), p=0.50),
        A.Perspective(scale=(0.04, 0.10), p=0.30),
        A.PadIfNeeded(img_size, img_size,
                      border_mode=cv2.BORDER_CONSTANT, value=(114,114,114), p=1.0),
        A.RandomCrop(img_size, img_size, p=1.0),

        # ── Photometric ─────────────────────────────────────────
        A.RandomBrightnessContrast(brightness_limit=0.40,
                                   contrast_limit=0.40, p=0.80),
        A.HueSaturationValue(hue_shift_limit=20,
                             sat_shift_limit=35,
                             val_shift_limit=25, p=0.60),
        A.ColorJitter(brightness=0.2, contrast=0.2,
                      saturation=0.2, hue=0.05, p=0.40),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8,8), p=0.20),

        # ── Blur / Noise ────────────────────────────────────────
        A.GaussianBlur(blur_limit=(3, 7), p=0.25),
        A.MotionBlur(blur_limit=9, p=0.20),
        gnoise,

        # ── Occlusion simulation ────────────────────────────────
        coarse,

    ], bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['class_labels'],
        min_visibility=0.30,   # drop boxes that become < 30% visible
        clip=True,
    ))

aug_pipeline = build_aug_pipeline()
print('Augmentation pipeline built.')
print(f'Transforms: {len(aug_pipeline.transforms)}')

## Visualise — Before vs After

In [ ]:
# Pick a random training image and show original vs 4 augmented versions
train_imgs = sorted(
    list(Path(f'{DATASET_DIR}/images/train').glob('*.jpg')) +
    list(Path(f'{DATASET_DIR}/images/train').glob('*.png'))
)
sample_path = random.choice(train_imgs)
sample_img  = load_image(sample_path)
sample_lbs  = load_labels(str(Path(f'{DATASET_DIR}/labels/train') / (sample_path.stem+'.txt')))

# Apply copy-paste first, then albumentations
cp_img, cp_labs = copy_paste(sample_img, sample_lbs, object_pool, num_paste=CP_PER_IMAGE)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Original  |  Copy-Paste  |  Aug 1-4', fontsize=13, fontweight='bold')

H, W = sample_img.shape[:2]

def show(ax, img, labels, title):
    ax.imshow(img)
    for lb in labels:
        cls = int(lb[0])
        x1,y1,x2,y2 = yolo_to_pixel(*lb[1:], img.shape[1], img.shape[0])
        color = [c/255 for c in CLASS_COLORS[cls]]
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                     linewidth=2, edgecolor=color, facecolor='none'))
        ax.text(x1, max(0,y1-4), CLASS_NAMES[cls], color=color,
                fontsize=7, fontweight='bold', backgroundcolor='white')
    ax.set_title(title, fontsize=9); ax.axis('off')

show(axes[0,0], sample_img, sample_lbs, f'Original ({len(sample_lbs)} boxes)')
show(axes[0,1], cp_img, cp_labs, f'+ Copy-Paste ({len(cp_labs)} boxes)')

for idx, ax in enumerate([axes[0,2], axes[1,0], axes[1,1], axes[1,2]]):
    bboxes      = [lb[1:] for lb in cp_labs]
    cls_labels  = [int(lb[0]) for lb in cp_labs]
    result = aug_pipeline(image=cp_img, bboxes=bboxes, class_labels=cls_labels)
    aug_lbs = [[c]+list(b) for c, b in zip(result['class_labels'], result['bboxes'])]
    show(ax, result['image'], aug_lbs, f'Augmented {idx+1} ({len(aug_lbs)} boxes)')

plt.tight_layout()
plt.savefig('../docs/figures/augmentation_preview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Preview saved to docs/figures/augmentation_preview.png')

## Generate Augmented Dataset

Creates `../dataset_augmented/` with the same train/val/test structure.
- **train**: original images + `AUGMENT_MULT × len(train)` copy-paste+augmented images
- **val / test**: copied as-is (never augmented — must reflect real performance)

In [ ]:
def setup_output_dirs(out_dir):
    for split in ['train','val','test']:
        os.makedirs(f'{out_dir}/images/{split}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{split}', exist_ok=True)

def copy_split(split, src, dst):
    """Copy val/test splits unchanged."""
    imgs = (list(Path(f'{src}/images/{split}').glob('*.jpg')) +
            list(Path(f'{src}/images/{split}').glob('*.png')))
    for img_path in imgs:
        shutil.copy(img_path, f'{dst}/images/{split}/{img_path.name}')
        lbl_path = Path(f'{src}/labels/{split}') / (img_path.stem + '.txt')
        if lbl_path.exists():
            shutil.copy(lbl_path, f'{dst}/labels/{split}/{lbl_path.name}')
    print(f'  Copied {len(imgs)} {split} images (unmodified)')

setup_output_dirs(OUTPUT_DIR)

# Copy val and test untouched
print('Copying val and test splits...')
copy_split('val',  DATASET_DIR, OUTPUT_DIR)
copy_split('test', DATASET_DIR, OUTPUT_DIR)

# Augment train split
print(f'\nGenerating augmented train split (x{AUGMENT_MULT})...')
train_imgs = sorted(
    list(Path(f'{DATASET_DIR}/images/train').glob('*.jpg')) +
    list(Path(f'{DATASET_DIR}/images/train').glob('*.png'))
)

# 1. Copy originals first
for p in train_imgs:
    shutil.copy(p, f'{OUTPUT_DIR}/images/train/{p.name}')
    lp = Path(f'{DATASET_DIR}/labels/train') / (p.stem+'.txt')
    if lp.exists():
        shutil.copy(lp, f'{OUTPUT_DIR}/labels/train/{lp.name}')

# 2. Generate augmented versions
generated = 0
pbar = tqdm(total=len(train_imgs)*AUGMENT_MULT, desc='Augmenting')

for aug_idx in range(AUGMENT_MULT):
    for img_path in train_imgs:
        img = load_image(img_path)
        if img is None:
            continue
        lbs = load_labels(str(Path(f'{DATASET_DIR}/labels/train') / (img_path.stem+'.txt')))

        # Step A: copy-paste
        aug_img, aug_lbs = copy_paste(img, lbs, object_pool, num_paste=CP_PER_IMAGE)

        # Step B: albumentations
        if aug_lbs:
            bboxes     = [lb[1:] for lb in aug_lbs]
            cls_labels = [int(lb[0]) for lb in aug_lbs]
            try:
                result   = aug_pipeline(image=aug_img, bboxes=bboxes, class_labels=cls_labels)
                aug_img  = result['image']
                aug_lbs  = [[c]+list(b) for c,b in zip(result['class_labels'], result['bboxes'])]
            except Exception:
                pass  # fallback: keep copy-paste result without albumentations

        stem    = f'{img_path.stem}_aug{aug_idx:02d}'
        out_img = f'{OUTPUT_DIR}/images/train/{stem}.jpg'
        out_lbl = f'{OUTPUT_DIR}/labels/train/{stem}.txt'

        cv2.imwrite(out_img, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        save_labels(aug_lbs, out_lbl)
        generated += 1
        pbar.update(1)

pbar.close()
print(f'\nGenerated {generated} augmented training images.')

## Dataset Statistics

In [ ]:
orig_train = len(list(Path(f'{DATASET_DIR}/images/train').glob('*.jpg')))
aug_train  = len(list(Path(f'{OUTPUT_DIR}/images/train').glob('*.jpg')))
aug_val    = len(list(Path(f'{OUTPUT_DIR}/images/val').glob('*.jpg')))
aug_test   = len(list(Path(f'{OUTPUT_DIR}/images/test').glob('*.jpg')))

# Count labels per class in augmented train
aug_class_counts = {n:0 for n in CLASS_NAMES}
for lbl_file in Path(f'{OUTPUT_DIR}/labels/train').glob('*.txt'):
    for lb in load_labels(lbl_file):
        aug_class_counts[CLASS_NAMES[int(lb[0])]] += 1

orig_class_counts = {n:0 for n in CLASS_NAMES}
for lbl_file in Path(f'{DATASET_DIR}/labels/train').glob('*.txt'):
    for lb in load_labels(lbl_file):
        orig_class_counts[CLASS_NAMES[int(lb[0])]] += 1

print('='*52)
print(f'{"":30s} {"Original":>8}  {"Augmented":>9}')
print('='*52)
print(f'  Train images    {"":14s} {orig_train:>8}  {aug_train:>9}  ({aug_train/orig_train:.1f}x)')
print(f'  Val images      {"":14s} {20:>8}  {aug_val:>9}')
print(f'  Test images     {"":14s} {21:>8}  {aug_test:>9}')
print('-'*52)
print('  Annotations per class (train):')
for name in CLASS_NAMES:
    o = orig_class_counts[name]
    a = aug_class_counts[name]
    bar = '#' * int(a/max(aug_class_counts.values())*25)
    print(f'    {name:20s}: {o:4d} → {a:5d}  {bar}')
print('='*52)

# Write data.yaml for the augmented dataset
yaml_content = f"""path: ../dataset_augmented
train: images/train
val: images/val
test: images/test
nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
with open(f'{OUTPUT_DIR}/data.yaml', 'w') as f:
    f.write(yaml_content)
print(f'\ndata.yaml written to {OUTPUT_DIR}/data.yaml')
print('\nTo train with the augmented dataset, change DATASET_DIR in')
print('02_phase2_finetuning.ipynb to:  DATASET_DIR = \'../dataset_augmented\'')

## How to Use With Training Notebook

In `02_phase2_finetuning.ipynb`, change one line in the config cell:

```python
# Before
DATASET_DIR = '../dataset'

# After
DATASET_DIR = '../dataset_augmented'
```

Everything else stays the same — the augmented dataset has identical folder structure.

---

**Expected impact:**
| Stage | Estimated mAP@0.5 |
|-------|-------------------|
| Original dataset (203 images) | ~0.45–0.50 |
| + Copy-paste + strong augmentation | ~0.58–0.68 |

The actual gain depends on class diversity in your original images.